In [27]:
import pandas as pd
import numpy as np
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_row",5)

In [9]:
df=pd.read_csv(r"C:\Users\hugod\Desktop\superstore.csv")
df


,Category,City,Country,Customer.ID,Customer.Name,Discount,Market,记录数,Order.Date,Order.ID,Order.Priority,Product.ID,Product.Name,Profit,Quantity,Region,Row.ID,Sales,Segment,Ship.Date,Ship.Mode,Shipping.Cost,State,Sub.Category,Year,Market2,weeknum
0,Office Supplies,Los Angeles,United States,LS-172304,Lycoris Saunders,0.0,US,1,2011-01-07 00:00:00.000,CA-2011-130813,High,OFF-PA-10002005,Xerox 225,9.3312,3,West,36624,19,Consumer,2011-01-09 00:00:00.000,Second Class,4.37,California,Paper,2011,North America,2
1,Office Supplies,Los Angeles,United States,MV-174854,Mark Van Huff,0.0,US,1,2011-01-21 00:00:00.000,CA-2011-148614,Medium,OFF-PA-10002893,"Wirebound Service Call Books, 5 1/2"" x 4""",9.2928,2,West,37033,19,Consumer,2011-01-26 00:00:00.000,Standard Class,0.94,California,Paper,2011,North America,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51288,Office Supplies,Los Angeles,United States,RM-196754,Robert Marley,0.2,US,1,2014-12-25 00:00:00.000,CA-2014-145219,Critical,OFF-BI-10001670,Vinyl Sectional Post Binders,33.9300,3,West,35917,90,Home Office,2014-12-26 00:00:00.000,First Class,15.95,California,Binders,2014,North America,52
51289,Office Supplies,Los Angeles,United States,FH-143654,Fred Hopkins,0.2,US,1,2014-12-26 00:00:00.000,CA-2014-121398,Medium,OFF-BI-10001718,GBC DocuBind P50 Personal Binding Machine,51.8238,3,West,37371,154,Corporate,2014-12-30 00:00:00.000,Standard Class,9.59,California,Binders,2014,North America,52


In [ ]:
df["Market"].value_counts()
df["Market2"].value_counts()

#Market 2 agrupates Canada and US under North America


Market2
APAC             11002
North America    10378
                 ...  
EMEA              5029
Africa            4587
Name: count, Length: 6, dtype: int64

In [16]:
df = df.drop(columns=["记录数", "weeknum", "Year"])

# Eliminamos columnas no necesarias:
# '记录数': 'record count'. Todos los valores son 1
# weeknum y Year: se pueden extraer después en SQL o Power BI


In [12]:
df = df.rename(columns={
    "Customer.ID": "CustomerID",
    "Customer.Name": "CustomerName",
    "Order.Date": "OrderDate",
    "Order.ID": "OrderID",
    "Order.Priority": "OrderPriority",
    "Product.ID": "ProductID",
    "Product.Name": "ProductName",
    "Row.ID": "RowID",
    "Ship.Date": "ShipDate",
    "Ship.Mode": "ShipMode",
    "Shipping.Cost": "ShippingCost",
    "Sub.Category": "SubCategory"
})
#Renaming of the columns with points in the name

In [17]:
df.duplicated().sum() #No duplicates found

0

In [ ]:
df.isnull().sum().to_frame().Tv #No Nulls found

,Category,City,Country,CustomerID,CustomerName,Discount,Market,记录数,OrderDate,OrderID,OrderPriority,ProductID,ProductName,Profit,Quantity,Region,RowID,Sales,Segment,ShipDate,ShipMode,ShippingCost,State,SubCategory,Year,Market2,weeknum
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [ ]:
(df["ShipDate"] < df["OrderDate"]).sum()  #No inconsistencies with date

0

In [22]:
checks = {
    "Sales <= 0": (df["Sales"] <= 0).sum(),
    "Quantity <= 0": (df["Quantity"] <= 0).sum(),
    "Discount fuera de [0, 1]": ((df["Discount"] < 0) | (df["Discount"] > 1)).sum(),
    "ShippingCost < 0": (df["ShippingCost"] < 0).sum()
}

for check, n in checks.items():
    print(f"{check}: {n}")

Sales <= 0: 1
Quantity <= 0: 0
Discount fuera de [0, 1]: 0
ShippingCost < 0: 0


In [ ]:
df[df["Sales"] <= 0]
df = df.drop(46158)   #This row has Sales=0 with Discount=0.8, it is not useful for analysis


,Category,City,Country,CustomerID,CustomerName,Discount,Market,OrderDate,OrderID,OrderPriority,ProductID,ProductName,Profit,Quantity,Region,RowID,Sales,Segment,ShipDate,ShipMode,ShippingCost,State,SubCategory,Market2
46158,Office Supplies,Houston,United States,ZC-219104,Zuschuss Carroll,0.8,US,2014-06-20 00:00:00.000,US-2014-102288,Medium,OFF-AP-10002906,Hoover Replacement Belt for Commercial Guardsm...,-1.11,1,Central,35398,0,Consumer,2014-06-24 00:00:00.000,Standard Class,0.01,Texas,Appliances,North America


In [ ]:
cols_texto = [
    "Category", "City", "Country", "CustomerName", "Market",
    "OrderPriority", "ProductName", "Region", "Segment",
    "ShipMode", "State", "SubCategory", "Market2"
]

for col in cols_texto:
    df[col] = df[col].astype(str).str.strip()

#Cleaning of empty spaces

In [50]:
# DATA MODEL TABLE CREATION

# 1. Create DimCustomer with one row per customer and CustomerID as primary key.
dim_customer = (
    df[[
        "CustomerID",
        "CustomerName",
        "Segment"
    ]]
    .drop_duplicates(subset=["CustomerID"], keep="first")
    .copy()
)

# Check that CustomerID is unique in DimCustomer.
print("DimCustomer rows:", len(dim_customer))
print("Duplicated CustomerID:", dim_customer["CustomerID"].duplicated().sum())

# 2. Create DimProduct with one row per product and ProductID as primary key.
dim_product = (
    df[[
        "ProductID",
        "ProductName",
        "Category",
        "SubCategory"
    ]]
    .drop_duplicates(subset=["ProductID"], keep="first")
    .copy()
)

# Check that ProductID is unique in DimProduct.
print("DimProduct rows:", len(dim_product))
print("Duplicated ProductID:", dim_product["ProductID"].duplicated().sum())

# 3. Create DimLocation with one row per unique location and LocationID as surrogate key.
dim_location = (
    df[[
        "City",
        "State",
        "Country",
        "Region",
        "Market",
        "Market2"
    ]]
    .drop_duplicates()
    .reset_index(drop=True)
    .copy()
)

# Create a surrogate key for each unique location.
dim_location["LocationID"] = range(1, len(dim_location) + 1)

# Reorder DimLocation columns.
dim_location = dim_location[[
    "LocationID",
    "City",
    "State",
    "Country",
    "Region",
    "Market",
    "Market2"
]]

# Check that LocationID is unique in DimLocation.
print("DimLocation rows:", len(dim_location))
print("Duplicated LocationID:", dim_location["LocationID"].duplicated().sum())

# 4. Add LocationID to the main dataframe so FactOrders can reference DimLocation.
df_with_location = df.merge(
    dim_location,
    on=[
        "City",
        "State",
        "Country",
        "Region",
        "Market",
        "Market2"
    ],
    how="left"
)

# 5. Create FactOrders with one row per order, using CustomerID and LocationID as foreign keys.
fact_orders = (
    df_with_location[[
        "OrderID",
        "CustomerID",
        "LocationID",
        "OrderDate",
        "ShipDate",
        "ShipMode",
        "OrderPriority"
    ]]
    .drop_duplicates(subset=["OrderID"], keep="first")
    .copy()
)

# Convert order date columns to datetime.
fact_orders["OrderDate"] = pd.to_datetime(fact_orders["OrderDate"], errors="coerce")
fact_orders["ShipDate"] = pd.to_datetime(fact_orders["ShipDate"], errors="coerce")

# Check that OrderID is unique in FactOrders.
print("FactOrders rows:", len(fact_orders))
print("Duplicated OrderID:", fact_orders["OrderID"].duplicated().sum())

# 6. Create FactOrderDetails with one row per order line and RowID as primary key.
fact_order_details = df[[
    "RowID",
    "OrderID",
    "ProductID",
    "Sales",
    "Quantity",
    "Discount",
    "Profit",
    "ShippingCost"
]].copy()

# Check that RowID is unique in FactOrderDetails.
print("FactOrderDetails rows:", len(fact_order_details))
print("Duplicated RowID:", fact_order_details["RowID"].duplicated().sum())

# 7. Export processed tables as CSV files.
dim_customer.to_csv("C:/Users/hugod/Desktop/dim_customer.csv", index=False)
dim_product.to_csv("C:/Users/hugod/Desktop/dim_product.csv", index=False)
dim_location.to_csv("C:/Users/hugod/Desktop/dim_location.csv", index=False)
fact_orders.to_csv("C:/Users/hugod/Desktop/fact_orders.csv", index=False)
fact_order_details.to_csv("C:/Users/hugod/Desktop/fact_order_details.csv", index=False)




DimCustomer rows: 4873
Duplicated CustomerID: 0
DimProduct rows: 10292
Duplicated ProductID: 0
DimLocation rows: 3819
Duplicated LocationID: 0
FactOrders rows: 25035
Duplicated OrderID: 0
FactOrderDetails rows: 51290
Duplicated RowID: 0


In [ ]:
# SYNTHETIC INCIDENTS TABLE CREATION

np.random.seed(42)
# 1. Create a base table with unique orders. Each incident will be linked to one order and one customer.
orders_base = df[["OrderID", "CustomerID", "OrderDate"]].drop_duplicates().copy()


# 2. Select a sample of orders that will have an incident.

incidents = orders_base.sample(frac=0.05, random_state=42).copy()

# 3. Create a unique identifier for each incident.
incidents["IncidentID"] = range(1, len(incidents) + 1)

incidents["IncidentType"] = np.random.choice(
    [
        "Delayed shipment",
        "Damaged product",
        "Wrong item",
        "Billing issue",
        "Return request",
        "Customer complaint"
    ],
    size=len(incidents),
    p=[0.30, 0.15, 0.10, 0.15, 0.20, 0.10]
)

# 5. Create the incident priority.
incidents["Priority"] = np.random.choice(
    ["Low", "Medium", "High", "Critical"],
    size=len(incidents),
    p=[0.35, 0.40, 0.20, 0.05]
)

# 6. Create the incident status.
incidents["Status"] = np.random.choice(
    ["Open", "In Progress", "Resolved", "Closed"],
    size=len(incidents),
    p=[0.15, 0.20, 0.45, 0.20]
)

# 7. Create the channel through which the incident was reported.
incidents["Channel"] = np.random.choice(
    ["Email", "Phone", "Web", "Chat"],
    size=len(incidents),
    p=[0.40, 0.25, 0.20, 0.15]
)

# 8. Create the incident opening date. The incident occurs between 1 and 14 days after the order date. Convertimos OrderDate a fecha real
incidents["OrderDate"] = pd.to_datetime(incidents["OrderDate"], errors="coerce")

# IncidentDate 1-15 days after 
incidents["IncidentDate"] = (
    incidents["OrderDate"] +
    pd.to_timedelta(
        np.random.randint(1, 15, size=len(incidents)),
        unit="D"
    )
)

# 9. Create the resolution date.  Only Resolved or Closed incidents will have a resolution date.
incidents["ResolutionDate"] = pd.NaT

mask_resolved = incidents["Status"].isin(["Resolved", "Closed"])

incidents.loc[mask_resolved, "ResolutionDate"] = (
    incidents.loc[mask_resolved, "IncidentDate"] +
    pd.to_timedelta(
        np.random.randint(1, 20, size=mask_resolved.sum()),
        unit="D"
    )
)

incidents = incidents[[
    "IncidentID",
    "OrderID",
    "CustomerID",
    "IncidentDate",
    "IncidentType",
    "Priority",
    "Status",
    "ResolutionDate",
    "Channel"
]]


print("Number of incidents created:", len(incidents))
display(incidents)
incidents.to_csv("C:/Users/hugod/Desktop/incidents.csv", index=False)

Number of incidents created: 1288


,IncidentID,OrderID,CustomerID,IncidentDate,IncidentType,Priority,Status,ResolutionDate,Channel
47725,1,CA-2011-140886,KW-165704,2011-10-07,Damaged product,Low,Closed,2011-10-24,Email
50069,2,CA-2013-149195,DM-135254,2013-09-14,Customer complaint,Low,Resolved,2013-09-20,Email
...,...,...,...,...,...,...,...,...,...
5192,1287,IN-2011-61582,AS-100451,2011-11-18,Delayed shipment,Low,Resolved,2011-11-19,Email
34490,1288,ES-2011-3805636,YC-218952,2011-05-14,Damaged product,Medium,In Progress,NaT,Email


In [ ]:
incidents[incidents["ResolutionDate"] < incidents["IncidentDate"]] #No Contradictory dates

,IncidentID,OrderID,CustomerID,IncidentDate,IncidentType,Priority,Status,ResolutionDate,Channel
